# Json File Reading
**Potential interview questions**:
1. What is json data and how to read it in spark ?
2. What if i have 3 keys in all line and 4 keys in one line ?
3. What is multiline and line-delimited json ?
4. Which one works faster -- multiline or line-delimited ?
5. How to convert nested json into spark dataframe ?
6. What will happens if i have corrupted json file or invalid json file ?

### JSON
- JavaScript Object Notation
- key value pair data
- It is semi-structured data

```json
{"id":1, "Name":"Manish", "Age":26},
{"id":2, "Name":"Vishva", "Age":25}
```

In [0]:
# %fs
# ls /FileStore/tables/

# BELOW ARE THE JSON FILES UPLOAD TO RUN THE CODE
# dbfs:/FileStore/tables/Multi_line_correct.json
# dbfs:/FileStore/tables/Multi_line_incorrect.json
# dbfs:/FileStore/tables/corrupted_json.json
# dbfs:/FileStore/tables/line_delimited_json.json
# dbfs:/FileStore/tables/line_delimited_json_extrafield.json

# How to read json file

In [0]:
# READING LINE DELIMITED JSON FILE
spark.read.format("json")\
    .option("inferSchema", "true")\
    .option("mode","PERMISSIVE")\
    .load("/FileStore/tables/line_delimited_json.json").show()

+---+--------+------+
|age|    name|salary|
+---+--------+------+
| 20|  Manish| 20000|
| 25|  Nikita| 21000|
| 16|  Pritam| 22000|
| 35|Prantosh| 25000|
| 67|  Vikash| 40000|
+---+--------+------+



##### Que 2 - What if i have 3 keys in all line and 4 keys in one line ?

##### ANS
##### 1 Schema Inference:
Spark infers the schema by sampling the JSON data. The extra key in one line will:
- Create a new column in the DataFrame.
- Rows without the extra key will have `null` in that column.
- Below is the example

In [0]:
# READING JSON WITH EXTRA FIELD
spark.read.format("json")\
    .option("inferSchema", "true")\
    .option("mode", "PERMISSIVE")\
    .load("/FileStore/tables/line_delimited_json_extrafield.json").show()

+---+------+--------+------+
|age|gender|    name|salary|
+---+------+--------+------+
| 20|  null|  Manish| 20000|
| 25|  null|  Nikita| 21000|
| 16|  null|  Pritam| 22000|
| 35|  null|Prantosh| 25000|
| 67|     M|  Vikash| 40000|
+---+------+--------+------+



##### Que 3 and 4 - What is multiline and line-delimited json ?
##### Line-Delimited JSON
- Format: Each line is a standalone JSON object.
```json
{"name": "Alice", "age": 25},  
{"name": "Bob", "age": 30}  
```
- **Spark Behavior:**
  - Default mode for `spark.read.json()`: Each line is passed as a seperate row.
  - **Faster Performance:**
    - Easily parallelized ( each  line = independent record)
    - Schema inference is efficient (Spark samples line without reading the entire file).
    - Splittable for distributed processing.

##### Multiline JOSN
- **format**: JOSN objects span multiple lines, often as a single JSON array:
- Spark Behavior:
  - Requires `multiLine=True` option:
  - `df = spark.read.format("json").option("multiline", "true").load("path/data.json)`
  - **Slower Performance**
    - Entire file must be loaded into memory to parse nasted structures.
    - Not splittable --> limits parallel processing.
    - Schema inference requires reading the whole file.

##### Key Takeaways:
- **Line-delimited JSON** is optimized for Spark’s distributed processing.
- **Multiline JSON** is slower and less scalable → avoid for large datasets.
- Always prefer **line-delimited** unless your data requires nested structures.



In [0]:
# READING SINGLE LINE / LINE DELIMITED JSON
spark.read.format("json")\
    .option("inferSchema", "true")\
    .option("mode", "PERMISSIVE")\
    .load("/FileStore/tables/line_delimited_json.json").show()

+---+--------+------+
|age|    name|salary|
+---+--------+------+
| 20|  Manish| 20000|
| 25|  Nikita| 21000|
| 16|  Pritam| 22000|
| 35|Prantosh| 25000|
| 67|  Vikash| 40000|
+---+--------+------+



In [0]:
# READ MULTILINE JSON
spark.read.format("json")\
    .option("inferSchema", "true")\
    .option("mode", "PERMISSIVE")\
    .option("multiline","true")\
    .load("/FileStore/tables/Multi_line_correct.json").show()

+---+--------+------+
|age|    name|salary|
+---+--------+------+
| 20|  Manish| 20000|
| 25|  Nikita| 21000|
| 16|  Pritam| 22000|
| 35|Prantosh| 25000|
| 67|  Vikash| 40000|
+---+--------+------+



In [0]:
# READ MULTILINE incorrect JSON
# in this json file we are not passing the multiline json in list that's why it is reading the first record. correct multiline json should passed in list `[]`.
# multiline json is list of dictionary.
spark.read.format("json")\
    .option("inferSchema", "true")\
    .option("mode", "PERMISSIVE")\
    .option("multiline","true")\
    .load("/FileStore/tables/Multi_line_incorrect.json").show()

+---+------+------+
|age|  name|salary|
+---+------+------+
| 20|Manish| 20000|
+---+------+------+



##### QUE 6 - What will happens if i have corrupted json file or invalid json file ?
- It will create a column with `_corrupt_record` and place the corrupted records in it.
- If record is not currupted then the column `_currupt_record` will be null for that record.

In [0]:
# READING THE CORRUPTED JSON FILE
spark.read.format("json")\
    .option("inferSchema", "true")\
    .option("mode", "PERMISSIVE")\
    .load("/FileStore/tables/corrupted_json.json").show(truncate=False)

+----------------------------------------+----+--------+------+
|_corrupt_record                         |age |name    |salary|
+----------------------------------------+----+--------+------+
|null                                    |20  |Manish  |20000 |
|null                                    |25  |Nikita  |21000 |
|null                                    |16  |Pritam  |22000 |
|null                                    |35  |Prantosh|25000 |
|{"name":"Vikash","age":67,"salary":40000|null|null    |null  |
+----------------------------------------+----+--------+------+



##### How to convert nested json into spark dataframe ?
- Need to learn

## END